### 0. 환경 설정 및 초기화

In [8]:
import os
import json
import asyncio
from concurrent.futures import ThreadPoolExecutor
import time
import numpy as np
from collections import Counter


# LangChain
from langchain.schema import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import ChatPromptTemplate

# LangGraph
from typing import TypedDict, List, Dict, Any, Optional, Tuple
from langgraph.graph  import StateGraph, END

from dotenv import load_dotenv
load_dotenv()

True

In [10]:
# LLM 초기화
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# 품질 임계값 설정
QUALITY_THRESHOLD = 0.6 # 검색 품질이 60% 이상이어야 함.
MAX_RETRIES = 2 # 최대 2번까지 재검색 시도

## 1. State 정의

- 근데 이렇게 속성을 어떻게 다 정의하는 걸까?

In [11]:
class CosmeticPlanningState(TypedDict):
    # 입력
    original_query: str                 # 사용자가 입력한 원본 쿼리(= 기획안 초안)
    
    # 쿼리 분석
    query_complexity: Dict[str, Any]    # 쿼리가 얼마나 복잡한지
    required_categories: List[str]      # 어떤 종류의 검색이 필요한지

    # Self-query 검색 결과
    extracted_filters: Dict[str, Any]   # 추출된 메타데이터 (브랜드, 가격 등)
    structured_query: str               # 필터를 제외한 순수 검색어
    
    # 쿼리 분해 결과
    decomposed_queries: List[Dict[str, Any]]    # 작은 검색 단위들

    # 검색 결과
    search_results: Dict[str, List[Document]]   # 각 검색 결과
    search_quality_scores: Dict[str, float]     # 각 검색 결과의 품질 점수
    retry_attempts: Dict[str, int]              # 각 검색 결과의 재검색 시도 횟수

    # 추출된 인사이트
    extracted_insights: Dict[str, Any]          # 추출된 인사이트

    # 최종 결과
    draft_proposal: str                         # 초안
    final_proposal: str                         # 최종 기획안
      
    # 메타데이터
    iteration_count: int                        # 전체 반복 횟수  
    feedback: Optional[str]                     # 사용자 피드백
        # 사용자는 개입 안할건데... 알아서 피드백 되도록 해야함. 그러면 기준을 정해야 함!

## 2. Node 정의

### 2-1. 쿼리 복잡도 분석 노드

In [ ]:
def analyze_query_complexity(state: CosmeticPlanningState) -> CosmeticPlanningState:
    """
    쿼리 복잡도를 분석하여 필요한 검색 카테고리 결정
    """
    prompt = ChatPromptTemplate.from_messages([
        (
            "system","""
쿼리를 분석하여 복잡도와 필요한 검색 카테고리를 결정하세요.

가능한 카테고리:
- target_audience: 타겟 고객층 분석 (거의 항상 필요)
- product_benefits: 제품 효능과 기능 (거의 항상 필요)
- differentiation: 차별화 전략
- brand_positioning: 브랜드 포지셔닝 (특정 브랜드 언급시 필요)
- marketing_strategy: 마케팅 전략 (마케팅/홍보 언급시 필요)
- price_analysis: 가격 분석 (가격대ㅔ 언급시 필요)
- ingredient_analysis: 성분 분석 (특성 성분 언급시 필요)
"""
        ),
        ("human", """쿼리: {query}
다음 형식으로 응답하세요:
{{
    "complexity_level": "low" | "medium" | "high",
    "key_aspects": ["식별된 핵심 요소들"],
    "re"

}}
        """)
    ])